#Speech Emotion Recognition with MLP Classifier



#Dataset
The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS)

---
Audio-only files

Audio-only files of all actors (01-24) are available as two separate zip files (~200 MB each):

Speech file (Audio_Speech_Actors_01-24.zip, 215 MB) contains 1440 files: 60 trials per actor x 24 actors = 1440.
Song file (Audio_Song_Actors_01-24.zip, 198 MB) contains 1012 files: 44 trials per actor x 23 actors = 1012.

Total=2452

---

---
Toronto emotional speech set (TESS)

---


There are a set of 200 target words were spoken in the carrier phrase "Say the word _' by two actresses (aged 26 and 64 years) and recordings were made of the set portraying each of seven emotions (anger, disgust, fear, happiness, pleasant surprise, sadness, and neutral). There are 2800 data points (audio files) in total.

The dataset is organised such that each of the two female actor and their emotions are contain within its own folder. And within that, all 200 target words audio file can be found. The format of the audio file is a WAV format


---



# Mount google drive



In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ejlok1/cremad")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cremad' dataset.
Path to dataset files: /kaggle/input/cremad


# Install following libraries

In [2]:
!pip install librosa soundfile numpy sklearn pyaudio

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [3]:
!pip install soundfile

In [4]:
!pip install resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.6 MB/s eta 0:00:00


# Make the necessary imports

In [5]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

Define a function extract_feature to extract the mfcc, chroma, and mel features from a sound file. This function takes 4 parameters- the file name and three Boolean parameters for the three features:

* mfcc: Mel Frequency Cepstral Coefficient, represents the short-term power spectrum of a sound
* chroma: Pertains to the 12 different pitch classes
* mel: Mel Spectrogram Frequency

In [6]:
def extract_feature(file_name, mfcc, chroma, mel):
    X, sample_rate = librosa.load(os.path.join(file_name), res_type='kaiser_fast')
    if chroma:
        stft=np.abs(librosa.stft(X))
    result=np.array([])
    if mfcc:
        mfccs=np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
        result=np.hstack((result, mfccs))
    if chroma:
        chroma=np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
        result=np.hstack((result, chroma))
    if mel:
        mel=np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T,axis=0)
        result=np.hstack((result, mel))
    return result

Now, let’s define a dictionary to hold numbers and the emotions available in the RAVDESS & TESS dataset, and a list to hold all 8 emotions- neutral,calm,happy,sad,angry,fearful,disgust,surprised.

In [7]:
# Emotions in the CREMA Dataset
# Anger, Disgust, Fear, Happy, Neutral, and Sad
emotions={
  'ANG':'anger',
  'HAP':'happy',
  'SAD':'sad',
  'FEA':'fear',
  'DIS':'disgust',
  'NEU':'neutral'
}
# Emotions to observe
observed_emotions=['neutral','happy','sad','anger','fear', 'disgust']

# Load the data and extract features for each sound file

In [8]:
def load_data(test_size=0.2):
    x,y=[],[]
    for file in glob.glob('/kaggle/input/cremad/AudioWAV/*.wav'):
        file_name=os.path.basename(file)
        emotion=emotions[file_name.split("_")[2]]
        if emotion not in observed_emotions:
            continue
        feature=extract_feature(file, mfcc=True, chroma=True, mel=True)
        x.append(feature)
        y.append(emotion)
    return train_test_split(np.array(x), y, test_size=test_size, train_size= 0.75,random_state=9)

# Split the Dataset
Time to split the dataset into training and testing sets! Let’s keep the test set 25% of everything and use the load_data function for this.

In [9]:
import resampy

In [10]:
# Split the dataset
import time
x_train,x_test,y_train,y_test=load_data(test_size=0.25)

/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


#Observe the shape of the training and testing datasets:

In [11]:
#Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

(5581, 1861)


# Number of features extracted.

In [12]:
# Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

Features extracted: 180


# MLP Classifier

In [13]:
# Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

#Fit/train the model.

In [14]:
# Train the model
model.fit(x_train,y_train)

MLPClassifier(alpha=0.01, batch_size=256, hidden_layer_sizes=(300,),
              learning_rate='adaptive', max_iter=500)

# Predict the accuracy of our model

Let’s predict the values for the test set. This gives us y_pred (the predicted emotions for the features in the test set).

In [15]:
# Predict for the test set
y_pred=model.predict(x_test)

To calculate the accuracy of our model, we’ll call up the accuracy_score() function we imported from sklearn. Finally, we’ll round the accuracy to 2 decimal places and print it out.

In [16]:
# Calculate the accuracy of our model
accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)
# Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 46.37%


#classification Report

In [17]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

       anger       0.70      0.63      0.67       328
     disgust       0.42      0.23      0.30       321
        fear       0.41      0.46      0.43       323
       happy       0.45      0.27      0.34       318
     neutral       0.38      0.71      0.50       276
         sad       0.47      0.52      0.49       295

    accuracy                           0.46      1861
   macro avg       0.47      0.47      0.45      1861
weighted avg       0.48      0.46      0.45      1861



# Confusion Matrix

In [18]:
from sklearn.metrics import confusion_matrix
matrix = confusion_matrix(y_test,y_pred)
print (matrix)

[[208  18  34  35  29   4]
 [ 22  73  40  25  84  77]
 [ 21  18 147  32  56  49]
 [ 46  38  59  85  79  11]
 [  0  14  23   7 196  36]
 [  0  11  55   4  71 154]]


#Thank You

In [19]:
!pip install skl2onnx onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 74.3 MB/s eta 0:00:00


In [25]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, x_train.shape[1]]))]

# 3. Convert to ONNX
onx = convert_sklearn(model, initial_types=initial_type, target_opset=15, options={'zipmap': False})

# 4. Save the model
with open("mlp_model.onnx", "wb") as f:
    f.write(onx.SerializeToString())
print("Model saved as mlp_model.onnx")

Model saved as mlp_model.onnx


In [30]:
import onnx

# Load your current model
input_model_path = "mlp_model.onnx"
output_model_path = "mlp_model_sentis_ready.onnx"

# Slice the model from the original input up to the numeric probabilities tensor
onnx.utils.extract_model(
    input_model_path,
    output_model_path,
    input_names=["float_input"],
    output_names=["probabilities"] # Cuts off ArgMax and ArrayFeatureExtractor
)

print("Cleaned model saved successfully!")

Cleaned model saved successfully!


In [26]:
!pip install onnxruntime

In [27]:
import onnxruntime as rt
import numpy as np

# Run inference with onnxruntime
sess = rt.InferenceSession("mlp_model.onnx")
input_name = sess.get_inputs()[0].name
# Convert x_test to float32 as expected by the ONNX model
pred_onx = sess.run(None, {input_name: x_test[:5].astype(np.float32)})[0]
print("Predictions:", pred_onx)

Predictions: ['neutral' 'anger' 'fear' 'happy' 'fear']


In [28]:
print(y_test[:5])

['sad', 'anger', 'fear', 'happy', 'sad']
